### Open AI Agents SDK — Functions and Tools

"The OpenAI Agents SDK is a lightweight framework for building AI agents."

This notebook extends the intro workflow by wrapping Python functions as **tools** with `@function_tool`, passing them to `Agent(tools=[...])`, and letting the model call them during `Runner.run`.

Documentation:
https://openai.github.io/openai-agents-python/

Tools documentation:
https://openai.github.io/openai-agents-python/tools/

Github Repo:
https://github.com/openai/openai-agents-python

In [ ]:
%pip install -q openai-agents python-dotenv wikipedia

In [ ]:
import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool, trace

##### Packages Overview

**openai-agents**  
For creating and orchestrating agents with function tools  
Classes:
  - Agent, Runner, trace, function_tool (decorator), FunctionTool

**python-dotenv**  
Loads environment variables from a `.env` file (e.g. `OPENAI_API_KEY`)

**wikipedia**  
Python wrapper for the Wikipedia API — used by our `wikipedia_search` tool
  - https://wikipedia.readthedocs.io/en/latest/code.html#api

In [ ]:
load_dotenv()

print("OpenAI API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

- https://aistudio.google.com/app/
- https://platform.openai.com/login

## Define Function Tools

Use `@function_tool` to wrap a Python function so the agent can call it. The SDK reads each function's **name**, **docstring**, and **type hints** to build the tool schema the model sees.

The tool below calls **live Wikipedia** via the [`wikipedia`](https://pypi.org/project/wikipedia/) package (requires internet).

In [ ]:
import time
import wikipedia

wikipedia.set_lang("en")


@function_tool
def wikipedia_search(query: str, max_results: int = 3, sentences: int = 5) -> str:
    """Search Wikipedia and return summaries for the top matching articles.

    Args:
        query: The search term or question.
        max_results: Number of articles to return (max 5).
        sentences: Approximate number of sentences per summary.
    """
    max_results = min(max_results, 5)

    try:
        titles = wikipedia.search(query, results=max_results)
    except Exception as e:
        return f"Search failed: {e}"

    if not titles:
        return f"No Wikipedia results for '{query}'."

    def fetch_page(title: str, retries: int = 3) -> tuple[str, str] | None:
        """Returns (summary, url) or None on failure."""
        for attempt in range(retries):
            try:
                page = wikipedia.page(title, auto_suggest=False)
                sentences_text = ". ".join(page.summary.split(". ")[:sentences])
                return sentences_text, page.url
            except wikipedia.DisambiguationError as e:
                title = e.options[0]
                continue
            except wikipedia.PageError:
                return None
            except ValueError as e:
                if "Expecting value" in str(e) and attempt < retries - 1:
                    time.sleep(1.5 ** attempt)
                    continue
                return None
            except Exception:
                return None
        return None

    parts = []
    for title in titles:
        result = fetch_page(title)
        if result:
            summary, url = result
            parts.append(f"Title: {title}\nURL: {url}\nSummary: {summary}")

    return "\n\n".join(parts) if parts else "No readable articles found."

## Simple Example with Tools

One agent with a single tool — the model decides when to call `wikipedia_search` based on the user's question.

In [ ]:
fact_finder = Agent(
    name="Fact Finder",
    instructions=(
        "You are a concise research assistant. "
        "Always call wikipedia_search before answering factual questions. "
        "If no search results are found try different variations of the query or topic that are more likely to yield results. "
        "Summarize the tool output in 3–5 bullets and include Wikipedia URLs."
    ),
    model="gpt-4o-mini",
    tools=[wikipedia_search],
)

result = await Runner.run(fact_finder, "World Cup")
display(Markdown(result.final_output))

### Overview of Workflow


**Agents**
- **Researcher** – gathers facts and details using `wikipedia_search`
- **Reporter** – produces a professional report and can fact-check with `wikipedia_search`

**Tools**
- `wikipedia_search` — search a topic and return short summaries with URLs

### Create Agents

In [ ]:
researcher_inst = f"You are a skilled and resourceful researcher. Your job is to deeply investigate any assigned topic, intelligently leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. "
"When using the wikipedia_search tool, if no search results are found try different variations of the query or topic that are more likely to yield results. "
"Your research should emphasize both recent developments and core facts, highlight significance and context, and clearly cite your sources when possible. Focus on accuracy, clarity, and actionable insight in your findings."

reporter_inst = f"You are a meticulous analyst renowned for your keen attention to detail. "
"You excel at transforming complex information into clear, concise, and actionable reports, making even the most intricate data accessible and understandable for your audience. "
"Leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. "

In [ ]:
researcher = Agent(
    name="Professional Researcher",
    instructions=researcher_inst,
    model="gpt-4o-mini",
    tools=[wikipedia_search],
)

In [ ]:
from agents.tool import WebSearchTool

reporter = Agent(
    name="Professional Reporter",
    instructions=reporter_inst,
    model="gpt-4.1-mini",
    tools=[WebSearchTool()] ## hosted function
)

#### Do Initial Research

In [ ]:
topic = "economics"

In [ ]:
with trace("research with function tools"):
    result = await Runner.run(
        researcher,
        f"Research the topic: {topic}. "
        "Use your tools to find interesting facts, people, dates, events, sources, etc..."
        "Output 8–10 detailed markdown bullets plus a Sources section. "
        "Only return markdown (no enclosing triple backticks).",
    )
    research_result = result.final_output

In [ ]:
display(Markdown(research_result))

https://platform.openai.com/logs?api=traces

#### Build the Report

In [ ]:
with trace("building the report"):
    result = await Runner.run(
        reporter,
        f"You are provided with the following research notes: "
        f"{research_result} "
        "For each bullet point, expand it into a clear and comprehensive report section. "
        "Retain factual accuracy from the original notes and preserve any attributions to sources. "
        "Where appropriate, enrich each section with relevant supporting details. "
        "Your output should be a full report structured by main topics. "
        "Use you web search tool to fact check information provided in the research notes. "
        'At the end, include a concise "Sources" list. '
        "Format the output as Markdown (but omit enclosing triple backticks).",
    )
    report_result = result.final_output

In [ ]:
display(Markdown(report_result))

#### Checkout the Traces

Tool calls appear in traces alongside model turns.

https://platform.openai.com/logs?api=traces